# VaultGuard — Exploratory Data Analysis


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = Path.cwd().resolve()

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'creditcard.csv'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH}. Download creditcard.csv and place it in data/raw/.')

sns.set_theme(style='whitegrid', context='notebook')
df = pd.read_csv(DATA_PATH)
print(f'Dataset path: {DATA_PATH}')
print(f'Dataset shape (rows, columns): {df.shape}')
display(df.head())
df.info()
display(df.describe().T)


In [ ]:
# Validate expected columns are present
expected_columns = {'Time', 'Amount', 'Class', *[f'V{i}' for i in range(1, 29)]}
missing_columns = expected_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'Dataset is missing expected columns: {sorted(missing_columns)}')

print('Missing values by column:')
display(df.isna().sum().sort_values(ascending=False).to_frame('missing_values'))
print(f'Duplicate rows: {df.duplicated().sum():,}')

class_counts = df['Class'].value_counts().sort_index()
class_percentages = (df['Class'].value_counts(normalize=True).sort_index() * 100).round(4)
class_summary = pd.DataFrame({'count': class_counts, 'percentage': class_percentages})
class_summary.index = class_summary.index.map({0: 'Legitimate', 1: 'Fraud'})
display(class_summary)


In [ ]:
# Count plot (saved)
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x='Class', hue='Class', palette=['#4C78A8', '#E45756'], legend=False, ax=ax)
ax.set(title='Legitimate vs Fraudulent Transactions', xlabel='Transaction class', ylabel='Number of transactions')
ax.set_xticks([0, 1], ['Legitimate', 'Fraud'])
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'class_count.png', dpi=300, bbox_inches='tight')
plt.show()


## Transaction amount and time

The `V1`–`V28` fields are anonymized PCA-transformed variables and should not be assigned speculative real-world meanings.

In [ ]:
# Amount histogram + capped boxplot (side-by-side) and save figure
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(data=df, x='Amount', hue='Class', bins=100, log_scale=(False, True), element='step', stat='count', common_norm=False, ax=axes[0])
axes[0].set_title('Transaction Amount Distribution')

sns.boxplot(data=df, x='Class', y='Amount', hue='Class', palette=['#4C78A8', '#E45756'], legend=False, ax=axes[1])
axes[1].set_ylim(0, 2500)
axes[1].set_xticks([0, 1], ['Legitimate', 'Fraud'])
axes[1].set_title('Transaction Amount by Class (capped at 2,500)')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'amount_and_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Fraud activity over time (saved)
fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(data=df.loc[df['Class'].eq(1)], x='Time', bins=75, color='#E45756', ax=ax)
ax.set(title='Fraud Activity Over Time', xlabel='Seconds elapsed since first transaction', ylabel='Number of fraudulent transactions')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fraud_activity_over_time.png', dpi=300, bbox_inches='tight')
plt.show()


## EDA conclusion

The severe class imbalance is the central modelling concern. Use a stratified split and select model thresholds with fraud-sensitive metrics—not accuracy alone.